In [1]:
# All paths in this notebook are relative to the repository root; anchor the working directory there
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Bus Level of Service per TAZ from the National GTFS — Bus and BRT (Metronit)

Builds a **level-of-service table for every study TAZ** from the Ministry of Transport's national GTFS feed (`Input/GTFS/israel-public-transportation.zip`, Git LFS; source https://gtfs.mot.gov.il/gtfsfiles/), for scheduled **bus** service and, separately, for the **Metronit BRT** lines. The BRT lines are identified by their route codes (מק"ט, the first field of `route_desc` in `routes.txt`). The codes supplied with the request (83001–83005) were checked against the feed: only 83001 is a Metronit line there; 83003 and 83005 belong to local lines in Kiryat Malachi and Arara BaNegev, and 83002 / 83004 do not exist. The Metronit lines in the feed (operator Superbus, `מטרונית` in the route names, Haifa endpoints) are **83001 line 1** (red, Hof HaCarmel CBS – Krayot CBS), **67002 line 2** (blue, Bat Galim station – Kiryat Ata), **67003 line 3** (green, Krayot CBS loop via Hadar), **62004 line 4** (purple, Hof HaCarmel CBS – Krayot CBS via the Carmel tunnels) and **52005 line 5 / 5א** (orange, Bat Galim station – Yagur terminal and the Yagur loop). The routes matched are printed below so the tagging can be audited.

**Method.** Stops are located in the 781 study TAZs (`Input/TAZ_North/TAZ_North.shp`, Israel TM Grid) by point-in-polygon. A representative **Tuesday** inside the feed's validity is chosen (the Tuesday on which the most services run). Trips of the services active that day are joined to their routes; `stop_times.txt` is read in chunks and reduced to the study-area stops and the active trips. Departures are windowed to the morning peak **06:00–09:00** and the peak hour **07:00–08:00** (departure time at the stop). Per TAZ, for all bus service and for BRT only:

| Metric | Definition |
|---|---|
| `stops_n` | stops in the TAZ with at least one departure on the day |
| `lines_n`, `lines` | distinct route codes (מק"ט) with a departure in the TAZ in 06:00–09:00 |
| `dep_0609`, `dep_0708` | distinct trips departing from a stop in the TAZ in the window (a trip that calls at several stops in the TAZ counts once) |
| `headway_0708_min` | 60 / `dep_0708` — the combined headway of everything that serves the TAZ, all lines and directions |
| `best_line_headway_0708_min` | the shortest headway of any single line-and-direction in the TAZ in the peak hour |
| `direct_reach_taz_n` | number of distinct TAZs reachable without a transfer on a trip departing the TAZ in 06:00–09:00 |
| `direct_reach_30min_taz_n` | the same within 30 scheduled minutes |
| `nearest_stop_m` | distance from the TAZ centroid to the nearest served stop |
| `stops_per_km2` | stop density |
| `LOS_frequency` | TCQSM frequency grade on the combined peak-hour headway: A ≤ 10 min, B ≤ 15, C ≤ 20, D ≤ 30, E ≤ 60, F > 60, "no service" |
| `LOS_best_line` | the same grade on the best single line-direction headway — the stricter view, closer to what a traveller to one destination experiences |
| `brt_access` | 1 where a Metronit line stops in the TAZ; `brt_lines` lists which |

A second product is a **direct-service skim between the 25 V2 areas** (`Output/gtfs/bus_direct_skim_area_v2.csv`): for every ordered pair of areas that a trip connects (a stop in the origin area followed by a stop in the destination area), the median scheduled in-vehicle time over the trips of the morning peak, the number of such trips in 07:00–08:00 and the implied headway, for all buses and for BRT only. It is the GTFS-based replacement for the fastest-path floor of step 26 (task E3): actual services, scheduled running time, and a wait derived from the timetable.

**Dry run.** When the GTFS archive is not present (only its LFS pointer), the notebook generates a small synthetic feed in the same format — two bus lines and one BRT line through real study TAZs — and runs every step on it so that the mechanics are exercised; the outputs are then written under `Output/gtfs/dry_run/` and are not results.

In [2]:
import io, zipfile, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
import shapefile
from pyproj import Transformer
from shapely.geometry import shape, Point
from shapely.strtree import STRtree

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
GTFS = 'Input/GTFS/israel-public-transportation.zip'
BRT_CODES = {83001: 'Metronit 1 (red)', 67002: 'Metronit 2 (blue)', 67003: 'Metronit 3 (green)', 62004: 'Metronit 4 (purple)', 52005: 'Metronit 5 (orange)'}   # codes as found in the feed (see the note above)
AM = (6 * 3600, 9 * 3600); PEAK = (7 * 3600, 8 * 3600)
def is_lfs_pointer(p): return (not os.path.exists(p)) or open(p, 'rb').read(40).startswith(b'version https://git-lfs')
DRY_RUN = is_lfs_pointer(GTFS)
OUT = 'Output/gtfs/dry_run' if DRY_RUN else 'Output/gtfs'; os.makedirs(OUT, exist_ok=True); os.makedirs('Output/figures', exist_ok=True)
print('DRY RUN on a synthetic feed — pull the GTFS archive with git lfs pull --include="Input/GTFS/*.zip" for the real run' if DRY_RUN else f'GTFS archive: {GTFS} ({os.path.getsize(GTFS) / 1e6:,.0f} MB)')

# ---- study TAZs: polygons, centroids, areas ----
tz = shapefile.Reader('Input/TAZ_North/TAZ_North'); polys = [shape(s.__geo_interface__) for s in tz.shapes()]; taz_ids = [int(r['TAZ_NUMBER']) for r in tz.records()]
tree = STRtree(polys); TAZ = pd.DataFrame({'TAZ': taz_ids, 'area_km2': [p.area / 1e6 for p in polys], 'cx': [p.centroid.x for p in polys], 'cy': [p.centroid.y for p in polys]}).drop_duplicates('TAZ').set_index('TAZ')
to_itm = Transformer.from_crs('EPSG:4326', 'EPSG:2039', always_xy=True)
xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); v2_areas = xl.parse('AreaCodes').set_index('AggCode'); v2_key = xl.parse('TazAgg').set_index('TAZ')['AggCode']
print(f"{len(TAZ)} study TAZs; {len(v2_key)} of them in the 25 V2 areas")

GTFS archive: Input/GTFS/israel-public-transportation.zip (181 MB)


781 study TAZs; 174 of them in the 25 V2 areas


In [3]:
def synthetic_feed():
    """A small GTFS in the national format for the dry run: bus lines 10001 / 10002 and BRT 83001 through real V2 TAZ centroids, one Tuesday."""
    to_wgs = Transformer.from_crs('EPSG:2039', 'EPSG:4326', always_xy=True)
    trunk = [201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212]; krayot = [104, 103, 102, 101]
    def taz_of(area): return int(v2_key[v2_key == area].index[0])
    seqs = {'10001': [taz_of(a) for a in trunk], '10002': [taz_of(a) for a in trunk[-3:] + krayot], '83001': [taz_of(a) for a in trunk[1:11]]}
    stops, routes, trips, st, cal = [], [], [], [], []
    sid = 0; stop_of = {}
    for code, tazs in seqs.items():
        for t in tazs:
            if t not in stop_of:
                sid += 1; lon, lat = to_wgs.transform(TAZ.loc[t, 'cx'], TAZ.loc[t, 'cy']); stop_of[t] = sid
                stops.append({'stop_id': sid, 'stop_code': 10000 + sid, 'stop_name': f'stop {sid}', 'stop_desc': '', 'stop_lat': lat, 'stop_lon': lon, 'location_type': 0, 'parent_station': '', 'zone_id': ''})
        for d in (0, 1):
            rid = f'{code}_{d}'; routes.append({'route_id': rid, 'agency_id': 3 if code == '83001' else 5, 'route_short_name': code[-2:], 'route_long_name': f'line {code}', 'route_desc': f'{code}-{d + 1}-#', 'route_type': 3, 'route_color': ''})
            seq = tazs if d == 0 else list(reversed(tazs)); hw = 10 if code == '83001' else 20
            for k, dep in enumerate(range(5 * 3600, 10 * 3600, hw * 60)):
                tid = f'{rid}_{k}'; trips.append({'route_id': rid, 'service_id': 'S1', 'trip_id': tid, 'trip_headsign': '', 'direction_id': d, 'shape_id': ''})
                for i, t in enumerate(seq):
                    sec = dep + i * 180; hh, mm, ss = sec // 3600, (sec % 3600) // 60, sec % 60
                    st.append({'trip_id': tid, 'arrival_time': f'{hh:02d}:{mm:02d}:{ss:02d}', 'departure_time': f'{hh:02d}:{mm:02d}:{ss:02d}', 'stop_id': stop_of[t], 'stop_sequence': i + 1, 'pickup_type': 0, 'drop_off_type': 0, 'shape_dist_traveled': i * 800})
    cal.append({'service_id': 'S1', 'sunday': 1, 'monday': 1, 'tuesday': 1, 'wednesday': 1, 'thursday': 1, 'friday': 0, 'saturday': 0, 'start_date': '20260901', 'end_date': '20261031'})
    agency = [{'agency_id': 3, 'agency_name': 'BRT operator', 'agency_url': '', 'agency_timezone': 'Asia/Jerusalem'}, {'agency_id': 5, 'agency_name': 'bus operator', 'agency_url': '', 'agency_timezone': 'Asia/Jerusalem'}]
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as z:
        for name, rows in [('stops.txt', stops), ('routes.txt', routes), ('trips.txt', trips), ('stop_times.txt', st), ('calendar.txt', cal), ('agency.txt', agency)]:
            z.writestr(name, pd.DataFrame(rows).to_csv(index=False))
    buf.seek(0); return zipfile.ZipFile(buf)
Z = synthetic_feed() if DRY_RUN else zipfile.ZipFile(GTFS)
def read(name, **kw): return pd.read_csv(Z.open(name), **kw)
print('feed files:', Z.namelist())

feed files: ['agency.txt', 'calendar.txt', 'fare_attributes.txt', 'fare_rules.txt', 'routes.txt', 'shapes.txt', 'stop_times.txt', 'stops.txt', 'translations.txt', 'trips.txt']


## 1. Stops in the study TAZs, the service day, the active trips

In [4]:
stops = read('stops.txt', usecols=['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon', 'location_type'] if not DRY_RUN else None, dtype={'stop_id': str})
stops = stops[stops['location_type'].fillna(0) == 0] if 'location_type' in stops else stops
stops = stops[stops['stop_lat'].between(32.3, 33.4) & stops['stop_lon'].between(34.8, 36.0)].copy()          # the study bbox before the point-in-polygon
stops[['x', 'y']] = [to_itm.transform(lo, la) for lo, la in zip(stops['stop_lon'], stops['stop_lat'])]
def taz_at(x, y):
    hit = tree.query(Point(x, y), predicate='within'); return taz_ids[hit[0]] if len(hit) else np.nan
stops['TAZ'] = [taz_at(x, y) for x, y in zip(stops['x'], stops['y'])]
stops = stops.dropna(subset=['TAZ']).astype({'TAZ': int}); stops['stop_id'] = stops['stop_id'].astype(str)
print(f"stops located in study TAZs: {len(stops):,} in {stops['TAZ'].nunique()} TAZs")
# ---- service day: the Tuesday within the feed with the most active services ----
cal = read('calendar.txt', dtype={'service_id': str}); cal['start'] = pd.to_datetime(cal['start_date'].astype(str)); cal['end'] = pd.to_datetime(cal['end_date'].astype(str))
cands = pd.date_range(cal['start'].min(), cal['end'].max(), freq='D'); cands = [d for d in cands if d.weekday() == 1]     # Monday=0 → Tuesday=1
n_active = {d: int(((cal['tuesday'] == 1) & (cal['start'] <= d) & (cal['end'] >= d)).sum()) for d in cands}
DAY = max(n_active, key=n_active.get); active_services = set(cal[(cal['tuesday'] == 1) & (cal['start'] <= DAY) & (cal['end'] >= DAY)]['service_id'])
if 'calendar_dates.txt' in Z.namelist():
    cd = read('calendar_dates.txt', dtype={'service_id': str}); cd = cd[pd.to_datetime(cd['date'].astype(str)) == DAY]
    active_services = (active_services | set(cd[cd['exception_type'] == 1]['service_id'])) - set(cd[cd['exception_type'] == 2]['service_id'])
print(f"service day: Tuesday {DAY.date()} ({len(active_services):,} active services; feed valid {cal['start'].min().date()} – {cal['end'].max().date()})")
# ---- routes, BRT flag, active trips ----
routes = read('routes.txt', dtype={'route_id': str, 'agency_id': str}); routes['route_code'] = pd.to_numeric(routes['route_desc'].astype(str).str.split('-').str[0], errors='coerce')
routes['route_dir'] = routes['route_desc'].astype(str).str.split('-').str[1]; routes['is_brt'] = routes['route_code'].isin(BRT_CODES)
agency = read('agency.txt', dtype={'agency_id': str}).set_index('agency_id')['agency_name'] if 'agency.txt' in Z.namelist() else pd.Series(dtype=str)
routes['agency'] = routes['agency_id'].map(agency)
trips = read('trips.txt', dtype={'route_id': str, 'service_id': str, 'trip_id': str}, usecols=['route_id', 'service_id', 'trip_id', 'direction_id'])
trips = trips[trips['service_id'].isin(active_services)].merge(routes[['route_id', 'route_code', 'route_dir', 'is_brt', 'agency', 'route_short_name']], on='route_id')
print(f"active trips on the day: {len(trips):,} on {trips['route_code'].nunique():,} route codes")
brt_routes = routes[routes['is_brt']].copy(); brt_routes['trips on the day'] = brt_routes['route_id'].map(trips.groupby('route_id').size()).fillna(0).astype(int)
brt_routes['line'] = brt_routes['route_code'].map(BRT_CODES)
print("BRT routes matched by code (audit — every row should be a Metronit route of the Haifa operator):")
print(brt_routes.sort_values(['route_code', 'route_desc'])[['route_id', 'line', 'agency', 'route_short_name', 'route_long_name', 'route_desc', 'trips on the day']].to_string(index=False))

stops located in study TAZs: 12,845 in 730 TAZs
service day: Tuesday 2026-06-02 (13,596 active services; feed valid 2026-05-22 – 2026-06-21)


active trips on the day: 123,828 on 3,287 route codes
BRT routes matched by code (audit — every row should be a Metronit route of the Haifa operator):
route_id                line  agency route_short_name                                                                      route_long_name route_desc  trips on the day
   34395 Metronit 5 (orange) סופרבוס               5א                                             ת. רכבת בת גלים-חיפה<->מסוף יגור-יגור-10  52005-1-0                 0
   34396 Metronit 5 (orange) סופרבוס               5א                                             מסוף יגור-יגור<->ת. רכבת בת גלים-חיפה-20  52005-2-0                 0
   34095 Metronit 5 (orange) סופרבוס                5                                                   מסוף יגור-יגור<->מסוף יגור-יגור-30  52005-3-0               109
   32346 Metronit 4 (purple) סופרבוס                4  ת. מרכזית חוף הכרמל/רציפים עירוני-חיפה<->מרכזית הקריות/הורדה מטרונית-קרית מוצקין-10  62004-1-0               142
   32348 

## 2. Stop times of the day in the study area, morning peak

In [5]:
def to_sec(s):
    p = s.str.split(':', expand=True).astype(float); return p[0] * 3600 + p[1] * 60 + p[2]
study_stop_ids = set(stops['stop_id']); active_trip_ids = set(trips['trip_id']); parts = []
with Z.open('stop_times.txt') as f:
    for chunk in pd.read_csv(f, usecols=['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence'], dtype={'trip_id': str, 'stop_id': str}, chunksize=2_000_000):
        c = chunk[chunk['stop_id'].isin(study_stop_ids) & chunk['trip_id'].isin(active_trip_ids)]
        if len(c): parts.append(c)
st = pd.concat(parts, ignore_index=True); st['dep'] = to_sec(st['departure_time']); st['arr'] = to_sec(st['arrival_time'])
st = st.merge(stops[['stop_id', 'TAZ']], on='stop_id').merge(trips[['trip_id', 'route_code', 'route_dir', 'direction_id', 'is_brt']], on='trip_id').sort_values(['trip_id', 'stop_sequence'])
am = st[st['dep'].between(*AM)].copy(); am['peak'] = am['dep'].between(*PEAK)
print(f"stop-time records in the study area on the day: {len(st):,} ({st['trip_id'].nunique():,} trips); in 06:00–09:00: {len(am):,} ({am['trip_id'].nunique():,} trips, BRT {am[am.is_brt]['trip_id'].nunique():,})")
# intermediates for step 30 (observed running times over the bus-speed network): the study-area stops and the full stop sequences of every trip that has a morning-peak departure in the study area
stops[['stop_id', 'stop_code', 'stop_name', 'x', 'y', 'TAZ']].to_csv(f'{OUT}/stops_study_area.csv', index=False, float_format='%.1f')
am_trips = set(am['trip_id']); st_out = st[st['trip_id'].isin(am_trips)][['trip_id', 'route_code', 'route_dir', 'direction_id', 'is_brt', 'stop_sequence', 'stop_id', 'TAZ', 'arr', 'dep']].copy()
st_out['peak'] = st_out['dep'].between(*PEAK); st_out.to_csv(f'{OUT}/stop_times_study_area_am_trips.csv.gz', index=False, compression='gzip')
print(f"saved {len(st_out):,} stop-time records of the {len(am_trips):,} morning-peak trips (all their study-area stops) and {len(stops):,} stops for step 30")

stop-time records in the study area on the day: 1,165,072 (33,499 trips); in 06:00–09:00: 213,076 (7,324 trips, BRT 267)


saved 259,438 stop-time records of the 7,324 morning-peak trips (all their study-area stops) and 12,845 stops for step 30


## 3. Level of service per TAZ — bus (all) and BRT

In [6]:
def los_grade(h):
    if pd.isna(h) or h == np.inf: return 'no service'
    return 'A' if h <= 10 else 'B' if h <= 15 else 'C' if h <= 20 else 'D' if h <= 30 else 'E' if h <= 60 else 'F'
def direct_reach(sub, max_min=None):
    """Per TAZ: distinct TAZs reachable downstream on the same trip (departing the TAZ in the window), optionally within max_min scheduled minutes."""
    reach = {}
    for tid, g in sub.groupby('trip_id', sort=False):
        tz_seq, dep_seq = g['TAZ'].values, g['dep'].values; arr_seq = g['arr'].values
        first = {}
        for i, t in enumerate(tz_seq):
            if t not in first: first[t] = i
        for t, i in first.items():
            down = tz_seq[i + 1:] if max_min is None else tz_seq[i + 1:][(arr_seq[i + 1:] - dep_seq[i]) <= max_min * 60]
            reach.setdefault(t, set()).update(x for x in down if x != t)
    return pd.Series({t: len(s) for t, s in reach.items()})
def taz_los(sub, label):
    g = sub.groupby('TAZ'); p = sub[sub['peak']].groupby('TAZ')
    out = pd.DataFrame({'stops_n': g['stop_id'].nunique(), 'lines_n': g['route_code'].nunique(), 'lines': g['route_code'].apply(lambda s: ' '.join(str(int(x)) for x in sorted(set(s.dropna())))),
                        'dep_0609': g['trip_id'].nunique(), 'dep_0708': p['trip_id'].nunique()}).reindex(TAZ.index)
    out[['stops_n', 'lines_n', 'dep_0609', 'dep_0708']] = out[['stops_n', 'lines_n', 'dep_0609', 'dep_0708']].fillna(0).astype(int); out['lines'] = out['lines'].fillna('')
    out['headway_0708_min'] = np.where(out['dep_0708'] > 0, 60 / out['dep_0708'].replace(0, np.nan), np.nan)
    line_dir = sub[sub['peak']].groupby(['TAZ', 'route_code', 'route_dir'])['trip_id'].nunique()
    out['best_line_headway_0708_min'] = (60 / line_dir).groupby('TAZ').min().reindex(TAZ.index)
    out['direct_reach_taz_n'] = direct_reach(sub).reindex(TAZ.index).fillna(0).astype(int)
    out['direct_reach_30min_taz_n'] = direct_reach(sub, 30).reindex(TAZ.index).fillna(0).astype(int)
    served = stops[stops['stop_id'].isin(sub['stop_id'].unique())]
    if len(served):
        from scipy.spatial import cKDTree
        d, _ = cKDTree(served[['x', 'y']].values).query(TAZ[['cx', 'cy']].values); out['nearest_stop_m'] = d
    else: out['nearest_stop_m'] = np.nan
    out['stops_per_km2'] = out['stops_n'] / TAZ['area_km2']
    out['LOS_frequency'] = out['headway_0708_min'].map(los_grade)
    out['LOS_best_line'] = out['best_line_headway_0708_min'].map(los_grade)          # the grade of the best single line-direction: the stricter, per-destination view
    out.columns = [f'{label}_{c}' for c in out.columns]
    return out
bus = taz_los(am, 'bus'); brt = taz_los(am[am['is_brt']], 'brt')
los = bus.join(brt); los['brt_access'] = (los['brt_stops_n'] > 0).astype(int)
los['brt_lines_named'] = los['brt_lines'].map(lambda s: '; '.join(BRT_CODES[int(c)] for c in s.split()) if s else '')
los['AggCode_V2'] = los.index.map(v2_key); los['AggAreaName_V2'] = los['AggCode_V2'].map(v2_areas['AggAreaName'])
los.insert(0, 'TAZ', los.index); los.to_csv(f'{OUT}/bus_los_taz.csv', index=False, float_format='%.2f')
summary = pd.DataFrame({'TAZs': [len(los), int((los['bus_dep_0708'] > 0).sum()), int(los['brt_access'].sum())],
                        'V2 TAZs (174)': [int(los['AggCode_V2'].notna().sum()), int(((los['bus_dep_0708'] > 0) & los['AggCode_V2'].notna()).sum()), int(((los['brt_access'] == 1) & los['AggCode_V2'].notna()).sum())]},
                       index=['all', 'with bus service in the peak hour', 'with BRT access'])
print(summary.to_string()); print("\nfrequency LOS of the TAZs — bus, combined peak-hour headway / best single line:"); print(pd.concat([los['bus_LOS_frequency'].value_counts().rename('combined'), los['bus_LOS_best_line'].value_counts().rename('best line')], axis=1).fillna(0).astype(int).to_string())
print("\nBRT access by V2 area (TAZs with a Metronit stop / TAZs in the area):")
print(los.dropna(subset=['AggCode_V2']).groupby('AggAreaName_V2').agg(taz=('TAZ', 'size'), brt_taz=('brt_access', 'sum'), brt_lines=('brt_lines', lambda s: ' '.join(sorted(set(' '.join(s).split()))))).query('brt_taz > 0').to_string())
los.sort_values('bus_dep_0708', ascending=False).head(12)[['TAZ', 'AggAreaName_V2', 'bus_stops_n', 'bus_lines_n', 'bus_dep_0708', 'bus_headway_0708_min', 'bus_best_line_headway_0708_min', 'bus_direct_reach_taz_n', 'bus_LOS_frequency', 'brt_access', 'brt_lines_named', 'brt_headway_0708_min']].round(1)

                                   TAZs  V2 TAZs (174)
all                                 781            174
with bus service in the peak hour   719            156
with BRT access                      60             41

frequency LOS of the TAZs — bus, combined peak-hour headway / best single line:
            combined  best line
A                639        215
no service        62         63
B                 31        239
D                 19         97
E                 17         61
C                 13        106

BRT access by V2 area (TAZs with a Metronit stop / TAZs in the area):
                        taz  brt_taz                brt_lines
AggAreaName_V2                                               
BatGalim-KiryatEliezer   10        4              67002 83001
Bazan-Hutsot              4        3  62004 67002 67003 83001
EinHayam                  1        1                    83001
Hamifrats                 6        2  62004 67002 67003 83001
Hamoshava                 3     

,TAZ,AggAreaName_V2,bus_stops_n,bus_lines_n,bus_dep_0708,bus_headway_0708_min,bus_best_line_headway_0708_min,bus_direct_reach_taz_n,bus_LOS_frequency,brt_access,brt_lines_named,brt_headway_0708_min
TAZ,,,,,,,,,,,,
1219,1219,Hamifrats,24,97,247,0.2,3.5,344,A,1,Metronit 4 (purple); Metronit 2 (blue); Metron...,0.8
1203,1203,TsometKiryatAta,15,44,236,0.3,2.9,271,A,1,Metronit 4 (purple); Metronit 2 (blue); Metron...,0.7
1517,1517,Matam-NeotPeres,22,31,210,0.3,5.5,184,A,1,Metronit 4 (purple); Metronit 1 (red),1.4
1218,1218,Hamifrats,4,39,201,0.3,1.0,82,A,0,,NaN
601,601,TsurShalom,27,29,195,0.3,2.6,182,A,1,Metronit 4 (purple); Metronit 3 (green); Metro...,0.9
2302,2302,NaN,7,30,194,0.3,1.0,53,A,0,,NaN
1718,1718,NaN,13,26,192,0.3,1.0,110,A,0,,NaN
1505,1505,BatGalim-KiryatEliezer,12,22,163,0.4,5.5,132,A,1,Metronit 2 (blue); Metronit 1 (red),1.5
1210,1210,Bazan-Hutsot,7,30,161,0.4,3.0,208,A,1,Metronit 4 (purple); Metronit 2 (blue); Metron...,0.7


## 4. Direct-service skim between the V2 areas (bus, BRT)

In [7]:
amv = am[am['TAZ'].isin(v2_key.index)].copy(); amv['area'] = amv['TAZ'].map(v2_key)
def area_skim(sub, label):
    rows = []
    for tid, g in sub.groupby('trip_id', sort=False):
        a_seq, dep_seq, arr_seq, pk, rc = g['area'].values, g['dep'].values, g['arr'].values, g['peak'].values, g['route_code'].values
        first = {}
        for i, a in enumerate(a_seq):
            if a not in first: first[a] = i
        for a, i in first.items():
            seen = set()
            for j in range(i + 1, len(a_seq)):
                b = a_seq[j]
                if b == a or b in seen: continue
                seen.add(b); rows.append((a, b, (arr_seq[j] - dep_seq[i]) / 60, bool(pk[i]), rc[i]))
    t = pd.DataFrame(rows, columns=['o', 'd', 'ivt_min', 'peak', 'route_code'])
    cols = ['o', 'd', f'{label}_ivt_min', f'{label}_trips_0609', f'{label}_trips_0708', f'{label}_headway_0708_min', f'{label}_best_line_headway_0708_min']
    if not len(t): return pd.DataFrame(columns=cols)
    s = t.groupby(['o', 'd']).agg(**{f'{label}_ivt_min': ('ivt_min', 'median'), f'{label}_trips_0609': ('ivt_min', 'size'), f'{label}_trips_0708': ('peak', 'sum')}).reset_index()
    s[f'{label}_headway_0708_min'] = 60 / s[f'{label}_trips_0708'].replace(0, np.nan)
    # task E3 / C4 (docs/NEXT_STEPS_HANDOVER_2026-09-23.md): the headway of the single busiest line-and-direction
    # actually serving the pair in the peak hour, beside the combined headway above that pools every line together
    line_trips = t[t['peak']].groupby(['o', 'd', 'route_code']).size()
    best_hw = (60 / line_trips).groupby(['o', 'd']).min().rename(f'{label}_best_line_headway_0708_min').reset_index()
    s = s.merge(best_hw, on=['o', 'd'], how='left')
    return s[cols]
skim = area_skim(amv, 'bus').merge(area_skim(amv[amv['is_brt']], 'brt'), on=['o', 'd'], how='left')
skim['o_name'] = skim['o'].map(v2_areas['AggAreaName']); skim['d_name'] = skim['d'].map(v2_areas['AggAreaName'])
skim = skim[['o', 'o_name', 'd', 'd_name'] + [c for c in skim.columns if c.startswith(('bus_', 'brt_'))]]
n_pairs = len(v2_key.groupby(v2_key).size()) ** 2 - len(v2_areas)
print(f"area pairs with a direct bus service in the morning peak: {len(skim)} of {n_pairs}; with a direct BRT service: {int(skim['brt_ivt_min'].notna().sum())}")
print(f"combined vs best-single-line headway (median, direct pairs): {skim['bus_headway_0708_min'].median():.1f} vs {skim['bus_best_line_headway_0708_min'].median():.1f} min")
# against the step-26 floor where it exists
floor_p = 'Output/gc/bus_ivt_network_area_v2.csv'
if os.path.exists(floor_p):
    floor = pd.read_csv(floor_p, index_col=0); floor.columns = floor.columns.astype(int)
    skim['step26_fastest_path_min'] = [floor.loc[o, d] if (o in floor.index and d in floor.columns) else np.nan for o, d in zip(skim['o'], skim['d'])]
    ratio = (skim['bus_ivt_min'] / skim['step26_fastest_path_min']).replace([np.inf, -np.inf], np.nan)
    print(f"scheduled direct in-vehicle time ÷ fastest-path floor: median {ratio.median():.2f} (p10 {ratio.quantile(.1):.2f}, p90 {ratio.quantile(.9):.2f}) on {int(ratio.notna().sum())} pairs")
skim.to_csv(f'{OUT}/bus_direct_skim_area_v2.csv', index=False, float_format='%.2f')
trunk_pairs = skim[skim['o'].between(201, 210) & skim['d'].between(201, 210)]
print(f"trunk area pairs (201–210) with a direct bus service: {len(trunk_pairs)} of 90; with a direct BRT service: {int(trunk_pairs['brt_ivt_min'].notna().sum())}")
skim.sort_values('bus_trips_0708', ascending=False).head(15).round(1)

area pairs with a direct bus service in the morning peak: 422 of 600; with a direct BRT service: 278
combined vs best-single-line headway (median, direct pairs): 3.8 vs 6.0 min
scheduled direct in-vehicle time ÷ fastest-path floor: median 1.12 (p10 0.53, p90 1.85) on 422 pairs
trunk area pairs (201–210) with a direct bus service: 82 of 90; with a direct BRT service: 72


,o,o_name,d,d_name,bus_ivt_min,bus_trips_0609,bus_trips_0708,bus_headway_0708_min,bus_best_line_headway_0708_min,brt_ivt_min,brt_trips_0609,brt_trips_0708,brt_headway_0708_min,brt_best_line_headway_0708_min,step26_fastest_path_min
291,212,TsometKiryatAta,211,Bazan-Hutsot,2.6,301,114,0.5,4.6,1.3,116.0,43.0,1.4,4.6,2.6
290,212,TsometKiryatAta,210,Hamifrats,8.0,257,102,0.6,4.6,11.0,110.0,43.0,1.4,4.6,15.3
267,211,Bazan-Hutsot,210,Hamifrats,4.6,259,96,0.6,4.6,8.4,111.0,43.0,1.4,4.6,18.5
268,211,Bazan-Hutsot,212,TsometKiryatAta,4.2,244,96,0.6,4.6,7.0,108.0,43.0,1.4,4.6,14.0
24,102,Kiryon,103,KiryatBialik Center,2.8,248,91,0.7,6.0,5.2,58.0,20.0,3.0,6.0,8.8
0,101,TsurShalom,102,Kiryon,5.0,221,87,0.7,5.5,5.0,59.0,21.0,2.9,5.5,5.4
1,101,TsurShalom,103,KiryatBialik Center,10.2,215,85,0.7,5.5,10.2,57.0,21.0,2.9,5.5,11.5
82,104,KiryatHaim,212,TsometKiryatAta,4.5,225,82,0.7,6.0,7.0,82.0,30.0,2.0,6.0,5.5
48,103,KiryatBialik Center,104,KiryatHaim,3.0,216,82,0.7,6.0,2.8,57.0,20.0,3.0,6.0,3.7
47,103,KiryatBialik Center,102,Kiryon,3.0,216,81,0.7,6.0,2.6,51.0,20.0,3.0,6.0,4.8


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(17, 9), facecolor='white')
grade_color = {'A': '#0d366b', 'B': '#2a78d6', 'C': '#86b6ef', 'D': '#f2c14e', 'E': '#eb6834', 'F': '#7f1e1e', 'no service': '#f0efec'}
study_polys = {t: p for t, p in zip(taz_ids, polys)}
def draw(ax, colors, title, legend_items):
    patches, cols = [], []
    for t, p in study_polys.items():
        geoms = [p] if p.geom_type == 'Polygon' else list(p.geoms)
        for gm in geoms: patches.append(MplPolygon(np.array(gm.exterior.coords), closed=True)); cols.append(colors.get(t, '#f0efec'))
    ax.add_collection(PatchCollection(patches, facecolor=cols, edgecolor='white', linewidth=0.2)); ax.set_xlim(195000, 240000); ax.set_ylim(730000, 765000); ax.set_aspect('equal')
    ax.set_title(title, color=INK, fontsize=11); ax.tick_params(labelsize=7, colors=INK2)
    for s in ax.spines.values(): s.set_color(AXIS)
    for lab, c in legend_items: ax.scatter([], [], color=c, s=60, label=lab)
    ax.legend(frameon=False, fontsize=8, loc='lower right')
draw(axes[0], {t: grade_color[g] for t, g in los['bus_LOS_frequency'].items()}, f'Bus frequency level of service, Tuesday {DAY.date()}, 07:00–08:00 (combined headway of all lines in the TAZ)', list(grade_color.items()))
brt_col = {t: (PURPLE if a else '#f0efec') for t, a in los['brt_access'].items()}
served_brt = stops[stops['stop_id'].isin(am[am.is_brt]['stop_id'].unique())]
draw(axes[1], brt_col, 'TAZs with Metronit (BRT) access, and the BRT stops', [('TAZ with a Metronit stop', PURPLE), ('no BRT stop', '#f0efec')])
axes[1].scatter(served_brt['x'], served_brt['y'], s=6, color=INK, zorder=3)
fig.suptitle('Bus level of service per TAZ from the national GTFS' + (' — DRY RUN on a synthetic feed' if DRY_RUN else ''), color=INK, fontsize=13, y=0.98)
plt.tight_layout(); fig.savefig(f'Output/figures/gtfs_bus_los_taz{"_dry_run" if DRY_RUN else ""}.png', dpi=150, bbox_inches='tight', facecolor='white'); plt.show()

## Findings

- **Feed and day.** National GTFS of 22 May 2026 (valid 22 May – 21 June); the representative day is **Tuesday 2 June 2026** (13,596 active services). 12,845 stops fall in 730 of the 781 study TAZs; 33,499 trips call at them on the day, 7,324 of them departing a study-area stop in 06:00–09:00 (267 Metronit trips).
- **The Metronit codes.** Of the five codes supplied with the request only 83001 is a Metronit line in this feed; 83003 and 83005 are local lines in Kiryat Malachi and Arara BaNegev, and 83002 / 83004 do not exist. The audit table above shows the lines actually used: 83001 (line 1), 67002 (line 2, Bat Galim – Kiryat Ata), 67003 (line 3, Krayot loop), 62004 (line 4, via the Carmel tunnels) and 52005 (line 5, Yagur; the 5א variants to Bat Galim carry no trips on the day). Peak-hour Metronit headways are 1.4–3 min combined at the Krayot and Haifa hubs.
- **Bus level of service per TAZ.** 719 of 781 TAZs have a bus departure in 07:00–08:00; 62 have none (59 without any stop, industrial or empty zones, 18 of them in the V2 areas — Bazan, port, Matam fringe, Kiryat Ata industry). On the **combined** headway 639 TAZs grade A, which says only that some bus passes often; on the **best single line** the picture is 215 A, 239 B, 106 C, 97 D, 61 E. Median centroid-to-stop distance is 194 m (72 TAZs beyond 1 km); the median TAZ reaches 47 other TAZs without a transfer, 30 of them within 30 scheduled minutes.
- **By V2 area.** The Haifa trunk areas are all A–B on the best line (median best-line headway 3–7 min; Hamifrats 97 lines, Bat Galim 52) with direct reach of 40–100 TAZs within 30 minutes. The branches are weaker: Kiryat Ata North / North-East best-line headways 18–20 min, 5 of their 14 TAZs without peak service, direct reach 10 TAZs; Shefaram 20 min; Hamovil 30 min; Nazareth 12 min with 2,438 peak-hour departures across its 38 TAZs, dense but local (reach 48). Tirat Carmel has only 8 lines at a 12-minute best headway.
- **BRT access.** 60 TAZs have a Metronit stop, 41 of them in the V2 areas across 20 of the 25 areas: every trunk area from Matam to Tsomet Kiryat Ata, the Krayot (Kiryat Haim, Kiryat Bialik Center, Kiryon, Tsur Shalom via lines 1 / 4), Kiryat Yam and Savyoney Yam (line 3) and Kiryat Ata North (line 2). Tirat Carmel, Shefaram, Hamovil, Nazareth and Kiryat Ata North-East have none.
- **The direct-service skim.** 422 of the 600 area pairs have a direct bus service in the morning peak, 278 a direct Metronit service; on the trunk 82 of 90 pairs (72 by Metronit) — the eight missing ones are Tirat Carmel to and from Bat Galim, Hamoshava, Lower City and Hamifrats, which need a change at Hof HaCarmel. Trunk pairs: median scheduled in-vehicle time 13.3 min at a combined headway of 1.7 min; by Metronit 10.0 min at 6 min. Scheduled direct in-vehicle time is 1.12 × the fastest-path floor of step 26 (median; p10 0.53, p90 1.85) — the floor was close on the trunk and far too fast on the long pairs (Tsur Shalom – Bat Galim 50 min scheduled against 31; Kiryat Haim – Lower City 26 against 16). From Nazareth the direct services to the Haifa trunk take 44–60 min at 8–30 minute headways, from Shefaram 56 min; the Kiryat Ata branch has direct service to only 15 of its 45 trunk pairs.
- **What this changes downstream.** Step 26 now takes the bus in-vehicle time, wait and stop access from this skim where a direct service exists (task E3), and carries the Metronit as its own mode on the pairs it connects. Open: transfer paths for the 178 pairs without a direct service, and observed (AVL) running times in place of the timetable.
- **Outputs.** `Output/gtfs/bus_los_taz.csv` (781 rows), `Output/gtfs/bus_direct_skim_area_v2.csv` (422 pairs), figure `gtfs_bus_los_taz.png`; the intermediates `stops_study_area.csv` and `stop_times_study_area_am_trips.csv.gz` for step 30; the earlier synthetic dry run remains under `Output/gtfs/dry_run/`.